# SG-VO — LoRA Online Adaptation Experiment
### ICRA Research Notebook

**Before running:** `Runtime → Change runtime type → T4 GPU`

Runs four conditions and compares them in a results table:

| # | Condition | Params updated |
|---|---|---|
| A | Offline VO (no adaptation) | 0 |
| B | Full online adaptation (paper) | 39M |
| C | LoRA-attention rank 8 | 21K (0.08%) |
| D | LoRA-attention + trigger 0.8 | 21K, ~50% of frames |

In [ ]:
# ── Cell 1: Check GPU ────────────────────────────────────────────────────────
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB')
else:
    print('⚠️  No GPU — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
# ── Cell 2: Clone Repo and Install Dependencies ──────────────────────────────
import os
if not os.path.exists('/content/SG-VO'):
    !git clone https://github.com/AreebaAzizRajput/SG-VO.git /content/SG-VO
else:
    %cd /content/SG-VO
    !git pull
%cd /content/SG-VO
!pip install -q einops path imageio blessings progressbar2 scikit-image tqdm gdown
print('✅ Done')

In [ ]:
# ── Cell 3: Download Pretrained Checkpoints ──────────────────────────────────
import os, glob, shutil
os.makedirs('checkpoints', exist_ok=True)

if not os.path.exists('checkpoints/exp_pose112_model_best.pth.tar'):
    print('Downloading pretrained models from Google Drive...')
    !gdown --folder 'https://drive.google.com/drive/folders/1twZsg2pxMLwcUUnFszBn1ryaEtuoEfs3' \
        -O checkpoints/ --quiet
    for f in glob.glob('checkpoints/**/*.pth.tar', recursive=True):
        dest = os.path.join('checkpoints', os.path.basename(f))
        if f != dest: shutil.move(f, dest)

!ls -lh checkpoints/*.pth.tar

In [ ]:
# ── Cell 4: Download Camera Intrinsics ───────────────────────────────────────
import os
os.makedirs('data/kitti_odom/sequences', exist_ok=True)

if not os.path.exists('data/kitti_odom/sequences/kitti_odom256_intrinsics/cam_09.txt'):
    print('Downloading camera intrinsics...')
    !gdown --folder 'https://drive.google.com/drive/folders/1n81QDHaG3lIxnxybl9I6knPHxPngTsD8' \
        -O data/kitti_odom/sequences/ --quiet

!ls data/kitti_odom/sequences/kitti_odom256_intrinsics/

In [ ]:
# ── Cell 5: Download KITTI Odometry Sequences (09 & 10) from GitHub Release ──
# Seq 09/10 image_2 frames (~1.5 GB total) are hosted as release assets on this
# repo (tag: kitti-data-v1), pre-packaged once via remotezip from the official
# KITTI odometry color zip. A plain wget of two tars replaces the old per-frame
# range-request fetch — a fresh Colab session is data-ready in ~2 minutes.
import os, glob, shutil, subprocess

DATASET_DIR = 'data/kitti_odom/sequences'
os.makedirs(DATASET_DIR, exist_ok=True)
for d in ['vo_results_offline','vo_results_full','vo_results_lora','vo_results_lora_trigger']:
    os.makedirs(d, exist_ok=True)

RELEASE = 'https://github.com/AreebaAzizRajput/SG-VO/releases/download/kitti-data-v1'

def seq_ok(seq):
    return len(glob.glob(f'{DATASET_DIR}/{seq}/image_2/*.png')) > 100

for seq in ['09', '10']:
    if seq_ok(seq):
        print(f'✅ Seq {seq} already present — skipping')
        continue
    tar = f'kitti_odom_{seq}.tar'
    print(f'Downloading {tar} ...')
    subprocess.run(['wget', '-q', '--show-progress', f'{RELEASE}/{tar}', '-O', f'/content/{tar}'], check=True)
    # Tar contains <seq>/image_2/*.png — extract straight into the sequences dir
    subprocess.run(['tar', '-xf', f'/content/{tar}', '-C', DATASET_DIR], check=True)
    os.remove(f'/content/{tar}')

# Final verification
SEQUENCES = [s for s in ['09','10'] if seq_ok(s)]
print(f'\nAvailable sequences: {SEQUENCES}')
for s in ['09','10']:
    n = len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png'))
    flag = '✅' if n > 100 else '❌'
    print(f'  Seq {s}: {n} images {flag}')
_,_,free = shutil.disk_usage('/'); print(f'Free disk: {free/1e9:.0f} GB')

In [ ]:
# ── Cell 6: Install cam.txt Intrinsics ───────────────────────────────────────
import os, shutil, glob

INTR_DIR    = 'data/kitti_odom/sequences/kitti_odom256_intrinsics'
DATASET_DIR = 'data/kitti_odom/sequences'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'{DATASET_DIR}/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    src  = f'{INTR_DIR}/cam_{seq}.txt'
    dest = f'{DATASET_DIR}/{seq}/image_2/cam.txt'
    if os.path.exists(src):
        shutil.copy(src, dest)
        print(f'✅ cam.txt → seq {seq}')
    else:
        print(f'⚠️  Missing intrinsics for seq {seq}: {src}')

In [ ]:
# ── Cell 7: LoRA Sanity Check ────────────────────────────────────────────────
# PoseResNet ALWAYS uses ResNet-18 regardless of --resnet-layers flag.
# (--resnet-layers only controls DispResNet, the depth network.)
# Using num_layers=50 here caused a channel mismatch: ResNet-50 outputs
# 2048-channel features but crossAttention expects 512.
import sys, torch
sys.path.insert(0, '/content/SG-VO')
from lora import inject_lora_pose_net, freeze_all, count_lora_params
from models import PoseResNet

pose_net = PoseResNet(num_layers=18, pretrained=False)  # always 18 for pose
n_total  = sum(p.numel() for p in pose_net.parameters())
freeze_all(pose_net)
inject_lora_pose_net(pose_net, rank=8, alpha=1.0, targets='attention')
n_lora = count_lora_params(pose_net)
print(f'LoRA params: {n_lora:,} / {n_total:,}  ({100*n_lora/n_total:.3f}%)')
print(f'Expected:    21,504 / 25,735,766  (0.084%)')

with torch.no_grad():
    out = pose_net(torch.randn(1, 36, 256, 832), torch.randn(1, 36, 256, 832))
print(f'Forward pass OK — output shape: {out.shape}')
print('✅ LoRA ready')

In [ ]:
# ── Cell 7b: Diagnostic — verify experiments will produce results ────────────
# Run this AFTER Cell 6 (cam.txt) and BEFORE the four experiment conditions.
# It runs Condition A (offline VO) directly with stderr captured, so any crash
# (ImportError, checkpoint mismatch, missing cam.txt) shows up here in ~2 min
# instead of being swallowed by the notebook's !python cells.
import subprocess, glob, os

# 1. Confirm KITTI data survived the session
for s in ['09', '10']:
    n = len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png'))
    print(f'Seq {s}: {n} frames  ({"OK" if n > 100 else "MISSING — re-run Cell 5"})')

# 2. Run offline VO directly with stderr captured (Condition A)
print('\n=== Running test_vo.py on Seq 09 (diagnostic) ===')
r = subprocess.run(
    ['python', 'test_vo.py',
     '--img-height', '256', '--img-width', '832', '--sequence', '09',
     '--pretrained-posenet', 'checkpoints/exp_pose112_model_best.pth.tar',
     '--dataset-dir', 'data/kitti_odom/sequences/',
     '--output-dir', 'vo_results_offline/'],
    capture_output=True, text=True)

print('\n--- return code:', r.returncode)
print('--- STDOUT (tail) ---\n', r.stdout[-2000:])
print('--- STDERR (tail) ---\n', r.stderr[-2000:])
wrote = os.path.exists('vo_results_offline/09.txt')
print('--- 09.txt written? ', wrote)
print('\n', '✅ Results are being written — proceed to the experiment cells.'
      if wrote else
      '❌ No results. Read STDERR above; do NOT run the long cells until this is fixed.')


## Condition A — Offline VO (baseline, no adaptation)

In [ ]:
# ── Cell 8: Offline VO ───────────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]
print(f'Running on sequences: {SEQUENCES}')

for seq in SEQUENCES:
    print(f'\n=== Offline VO — Seq {seq} ===')
    !python test_vo.py \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir vo_results_offline/

print('\n=== Offline results ===')
!python kitti_eval/eval_odom.py --result=vo_results_offline/ --align=7dof

## Condition B — Full Online Adaptation (paper baseline)

In [ ]:
# ── Cell 9: Full Online VO ───────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== Full Online VO — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_full/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border

print('\n=== Full online results ===')
!python kitti_eval/eval_odom.py --result=vo_results_full/ --align=7dof

## Condition C — LoRA-Attention Adaptation (new contribution)

In [ ]:
# ── Cell 10: LoRA Online VO ──────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== LoRA Online VO — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_lora/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention

print('\n=== LoRA results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora/ --align=7dof

## Condition D — LoRA + Trigger (efficiency variant)

In [ ]:
# ── Cell 11: LoRA + Trigger ──────────────────────────────────────────────────
import glob
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'data/kitti_odom/sequences/{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== LoRA + Trigger — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_lora_trigger/ \
        --epochs 2 --lr 1e-4 \
        --sequence-length 3 \
        --resnet-layers 50 \
        --select-best 1 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --lora-rank 8 \
        --lora-targets attention \
        --cusum-h 16

print('\n=== LoRA + Trigger results ===')
!python kitti_eval/eval_odom.py --result=vo_results_lora_trigger/ --align=7dof

### Probe-only loss recording — trigger design tool
Records the per-frame photometric loss **without any adaptation** so trigger
designs (spike / dual-EMA / CUSUM) can be simulated offline on real traces
before spending GPU time. Download lands as `probe_losses.zip`.

In [ ]:
# ── Cell 11b: Probe-only loss trace recording (no adaptation) ────────────────
import glob, zipfile, os
DATASET_DIR = 'data/kitti_odom/sequences/'
POSE_NET    = 'checkpoints/exp_pose112_model_best.pth.tar'
DISP_NET    = 'checkpoints/dispnet112_model_best.pth.tar'
os.makedirs('vo_results_probe', exist_ok=True)
SEQUENCES   = [s for s in ['09','10']
               if len(glob.glob(f'{DATASET_DIR}{s}/image_2/*.png')) > 100]

for seq in SEQUENCES:
    print(f'\n=== Probe-only loss recording — Seq {seq} ===')
    !python test_vo_online.py \
        -p 1 -s 0.1 -c 0.5 \
        --img-height 256 --img-width 832 \
        --sequence {seq} \
        --pretrained-posenet {POSE_NET} \
        --pretrained-disp    {DISP_NET} \
        --dataset-dir {DATASET_DIR} \
        --output-dir  vo_results_probe/ \
        --sequence-length 3 \
        --resnet-layers 50 \
        --with-mask 1 --with-auto-mask 1 \
        --padding-mode border \
        --probe-only

with zipfile.ZipFile('/content/probe_losses.zip', 'w') as z:
    for f in glob.glob('vo_results_probe/probe_losses_*.txt'):
        z.write(f)
from google.colab import files
files.download('/content/probe_losses.zip')

## Results Comparison Table

In [ ]:
# ── Cell 12: Comparison Table ────────────────────────────────────────────────
import subprocess, re, os

PAPER = {
    'offline': {'09': (7.08, 2.48), '10': (8.72, 3.11)},
    'online':  {'09': (5.21, 1.93), '10': (6.74, 2.57)},
}

def parse_eval(result_dir):
    out = subprocess.run(
        ['python', 'kitti_eval/eval_odom.py',
         f'--result={result_dir}', '--align=7dof'],
        capture_output=True, text=True).stdout
    results = {}
    for seq in ['09', '10']:
        m = re.search(rf'{seq}.*?([0-9]+\.[0-9]+).*?([0-9]+\.[0-9]+)', out)
        results[seq] = (float(m.group(1)), float(m.group(2))) if m else (None, None)
    return results

experiments = [
    ('A  Offline VO',         'vo_results_offline'),
    ('B  Full online (paper)', 'vo_results_full'),
    ('C  LoRA-attn rank 8',    'vo_results_lora'),
    ('D  LoRA + trigger 0.8',  'vo_results_lora_trigger'),
]

W = 28
print(f"{'Condition':<{W}} | {'Seq 09':^19} | {'Seq 10':^19}")
print(f"{'':<{W}} | {'t_err%':>9} {'r°/100m':>8} | {'t_err%':>9} {'r°/100m':>8}")
print('-' * 72)
for label, key in [('  Paper offline', 'offline'), ('  Paper online', 'online')]:
    r09, r10 = PAPER[key]['09'], PAPER[key]['10']
    print(f"{label:<{W}} | {r09[0]:9.2f} {r09[1]:8.2f} | {r10[0]:9.2f} {r10[1]:8.2f}")
print('-' * 72)
for name, d in experiments:
    if not os.path.exists(d):
        continue
    r = parse_eval(d)
    def fmt(v): return f'{v:9.2f}' if v else '      N/A'
    t09, r09 = r.get('09', (None, None))
    t10, r10 = r.get('10', (None, None))
    print(f"{name:<{W}} | {fmt(t09)} {fmt(r09)} | {fmt(t10)} {fmt(r10)}")

## Trajectory Visualization

In [ ]:
# ── Cell 13: Trajectory Plots ────────────────────────────────────────────────
exec(open('scripts/colab_eval_viz.py').read())

In [ ]:
# ── Cell 14: Download All Results ────────────────────────────────────────────
import zipfile, os
from google.colab import files

with zipfile.ZipFile('/content/sgvo_lora_results.zip', 'w') as z:
    for d in ['vo_results_offline', 'vo_results_full',
              'vo_results_lora', 'vo_results_lora_trigger']:
        if not os.path.exists(d): continue
        for f in os.listdir(d):
            z.write(os.path.join(d, f))

files.download('/content/sgvo_lora_results.zip')